In [3]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import gymnasium as gym
from IPython.display import Video

In [29]:
# initialize our policy network
class PolicyNetwork(nn.Module): 

    def __init__(self, input_features=8, output_features=4, hidden_features=128):
        super(PolicyNetwork, self).__init__()

        self.fc1 = nn.Linear(input_features, hidden_features)
        self.fc2 = nn.Linear(hidden_features, hidden_features)
        self.fc3 = nn.Linear(hidden_features, output_features)

    def forward(self, x): 

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        pi = F.softmax(x, dim=-1)

        return pi

In [45]:
# Compute returns for the reward trajectory 
def compute_returns(reward_trajectory, gamma): 

    G = 0
    returns = []

    # of shape (state, next_state, action, reward, done)
    for reward in reversed(reward_trajectory):

        G = reward + gamma * G

        # insert at the beginning since we're walking the trajectory backwards 
        returns.insert(0, G)

    # return in a tensor form since we'll use this for the loss
    return torch.tensor(returns, dtype=torch.float32)


In [48]:
# Training - where we run the game, calculate returns, calculate loss - entropy, then step the model
def train(
    env,
    entropy_weight=0.01,
    gamma=0.99,
    learning_rate=0.0005,
    input_features=8,
    output_features=4,
    hidden_features=128,
    num_episodes=5000,
    running_avg_steps=25,
    log_freq=3,
    device="mps",
    ): 

    # initialize our policy
    policy = PolicyNetwork(input_features, output_features, hidden_features).to(device)

    optimizer = optim.Adam(policy.parameters(), lr = learning_rate)

    log = {
        "scores": [],
        "running_avg_scores": []
    }

    episode_rewards = []

    for i in range(num_episodes): 

        state, _ = env.reset()
        
        log_probs = []
        entropies = []
        rewards = []
        done = False
        
        while not done: 

            state = torch.tensor(state, dtype=torch.float32).unsqueeze(0).to(device)
            possible_actions = policy(state) # outputs probability of taking said action, shape of (1,4)
            selected_action = torch.multinomial(possible_actions, 1).item() # returns index, action chosen according to output probability
            log_action_probs = torch.log(possible_actions[0, selected_action]) # find the log probability that we selected the action that we selected

            log_probs.append(log_action_probs)

            next_state, reward, terminal, truncated, _ = env.step(selected_action) # you need to pass in an integer not 
            done = terminal or truncated

            rewards.append(reward)

            # compute our entropy = -log(probs of each action) * probs of each action
            entropy = -torch.sum(torch.log(possible_actions+1e-8)*possible_actions)
            entropies.append(entropy)

            state = next_state
        
        returns = compute_returns(rewards, gamma).to(device)

        # normalize the returns 
        returns = (returns - returns.mean()) / (returns.std() + 1e-8)

        # compute our loss which is -objective - entropy
        loss = -(torch.sum(torch.stack(log_probs)*returns)) - entropy_weight * torch.sum(torch.stack(entropies))

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm(policy.parameters(), max_norm = 1.0) # max norm does L2 norm clipping it so that it equals 1
        optimizer.step()

        total_rewards = np.sum(rewards)

        log["scores"].append(total_rewards)
        running_avg_scores = np.mean(log["scores"][-running_avg_steps:])
        log["running_avg_scores"].append(running_avg_scores)

        if i % log_freq == 0:
            print(f"Episode {i}, Total Reward: {total_rewards}, Average Reward: {running_avg_scores}")

        if running_avg_scores >= 200: 
            print("Training complete")
            break

    return policy, log


In [ ]:
### Play Game ###
env = gym.make("LunarLander-v3", render_mode="rgb_array")
policy, log = train(env, device="mps")

/var/folders/jl/ppm07cc55jgfvx0gzqfyhtrw0000gn/T/ipykernel_25131/1003265522.py:67: FutureWarning: `torch.nn.utils.clip_grad_norm` is now deprecated in favor of `torch.nn.utils.clip_grad_norm_`.
  torch.nn.utils.clip_grad_norm(policy.parameters(), max_norm = 1.0) # max norm does L2 norm clipping it so that it equals 1


Episode 0, Total Reward: -219.73564894352924, Average Reward: -219.73564894352924
Episode 3, Total Reward: -140.32451848060148, Average Reward: -128.88573606097788
Episode 6, Total Reward: -116.19354924159444, Average Reward: -127.40657090456921
Episode 9, Total Reward: -137.84009263662693, Average Reward: -121.08557502424746
Episode 12, Total Reward: -595.7787102060411, Average Reward: -156.76521794139734
Episode 15, Total Reward: -353.4355591500233, Average Reward: -177.4248672395297
Episode 18, Total Reward: -325.5929555047792, Average Reward: -180.3373936218196
Episode 21, Total Reward: -146.6705983139773, Average Reward: -183.9464001273983
Episode 24, Total Reward: -173.77923274523266, Average Reward: -174.9867841299935
Episode 27, Total Reward: -11.631676888873884, Average Reward: -168.1240998279283
Episode 30, Total Reward: -231.34568447752739, Average Reward: -173.65246379723328
Episode 33, Total Reward: -64.56830519580461, Average Reward: -184.32825861321612
Episode 36, Total 